# Standalone LCO Observation Scraper to Google Drive

This notebook downloads exoplanet observation sequences from the Las Cumbres Observatory Science Archive directly into Google Drive. It is standalone: it does not import `observation_scraper.py`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip -q install requests astropy photutils astroquery matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 56.0 MB/s eta 0:00:00


In [ ]:
import csv
import getpass
import json
import os
import re
import shutil
import subprocess
import time
import warnings
from collections import OrderedDict, defaultdict
from pathlib import Path

import numpy as np
import requests
from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.stats import sigma_clipped_stats
from astropy.time import Time
from astropy.wcs import FITSFixedWarning, WCS
from photutils.aperture import CircularAperture, aperture_photometry
from photutils.detection import DAOStarFinder

ARCHIVE_API_ROOT = 'https://archive-api.lco.global'
FRAMES_URL = f'{ARCHIVE_API_ROOT}/frames/'
NASA_EXOPLANET_TAP_SYNC_URL = 'https://exoplanetarchive.ipac.caltech.edu/TAP/sync'

DEFAULT_TARGETS = [
    'WASP-43', 'WASP-36', 'HD 189733', 'TrES-3', 'HAT-P-32',
    'WASP-12', 'WASP-33', 'WASP-52', 'WASP-69', 'WASP-80',
    'WASP-103', 'WASP-107', 'HAT-P-3', 'HAT-P-11', 'HAT-P-12',
    'HAT-P-23', 'HAT-P-36', 'Qatar-1', 'XO-2', 'KELT-16',
    'GJ 1214', 'GJ 436', 'HD 209458', 'HD 80606', 'HAT-P-1',
    'HAT-P-7', 'HAT-P-13', 'HAT-P-20', 'HAT-P-30', 'HAT-P-37',
    'KELT-1', 'KELT-3', 'KELT-7', 'KELT-9', 'KELT-20',
    'TrES-2', 'TrES-4', 'XO-1', 'XO-3', 'WASP-4',
    'WASP-5', 'WASP-6', 'WASP-10', 'WASP-14', 'WASP-18',
    'WASP-19', 'WASP-24', 'WASP-32', 'WASP-46', 'WASP-48',
    'WASP-76',
]

FRAME_CSV_FIELDS = [
    'id', 'filename', 'basename', 'target_name', 'observation_id',
    'request_id', 'observation_date', 'proposal_id', 'site_id',
    'telescope_id', 'instrument_id', 'configuration_type',
    'reduction_level', 'primary_optical_element', 'exposure_time',
    'public_date', 'url',
]

METADATA_SUBDIR = '_metadata'
CALIBRATION_CONFIGURATION_TYPES = {'BIAS', 'DARK', 'SKYFLAT'}
CALIBRATION_SUBDIRS = {'BIAS': 'biases', 'DARK': 'darks', 'SKYFLAT': 'flats'}

MIN_SCIENCE_FRAMES = 100
MAX_SCIENCE_FRAMES = 300
TARGET_MATCH_MAX_DISTANCE_PX = 15.0
MIN_EDGE_DISTANCE_PX = 50
MIN_COMPARISON_STARS = 3
MAX_COMPARISON_STARS = 10
MIN_FRAME_DETECTION_FRACTION = 0.80
MIN_TARGET_VALIDATION_FRACTION = 0.80
E91_VALIDATION_FRAME_COUNT = 1
COMPARISON_APERTURE_RADIUS_PX = 6.0
MIN_COMPARISON_ISOLATION_PX = 15.0
MIN_TARGET_COMPARISON_DISTANCE_PX = 20.0
MAX_COMPARISON_ROBUST_RMS = 0.05
SATURATION_FRACTION = 0.90
MIN_COMPARISON_TARGET_FLUX_RATIO = 0.05
MAX_COMPARISON_TARGET_FLUX_RATIO = 5.0

In [ ]:
# Optional token. Public archive data can usually be downloaded without one.
try:
    from google.colab import userdata
    token = userdata.get('LCO_ARCHIVE_TOKEN')
except Exception:
    token = None

if not token:
    entered = getpass.getpass('LCO archive token (optional; press Enter for public-only): ')
    token = entered or None

if token:
    os.environ['LCO_ARCHIVE_TOKEN'] = token

def archive_headers(api_token=None):
    headers = {'Accept': 'application/json'}
    token = api_token or os.getenv('LCO_ARCHIVE_TOKEN')
    if token:
        headers['Authorization'] = f'Token {token}'
    return headers

In [ ]:
def request_json(session, url, headers, params=None, timeout=60, retries=3, backoff_seconds=3.0):
    response = None
    last_error = None
    for attempt in range(1, retries + 1):
        try:
            response = session.get(url, headers=headers, params=params, timeout=timeout)
            break
        except (requests.Timeout, requests.ConnectionError) as error:
            last_error = error
            if attempt >= retries:
                request_url = requests.Request('GET', url, params=params).prepare().url
                raise RuntimeError(
                    f'LCO archive request timed out after {retries} attempts: {request_url}. '
                    f'Last error: {error}'
                ) from error
            print(f' timeout; retrying in {backoff_seconds:.0f}s ({attempt}/{retries})', flush=True)
            time.sleep(backoff_seconds)

    if response is None:
        raise RuntimeError(f'LCO archive request failed before receiving a response: {last_error}')

    try:
        response.raise_for_status()
    except requests.HTTPError as error:
        body = response.text[:2000].replace('\n', ' ')
        raise RuntimeError(
            f'LCO archive request failed with HTTP {response.status_code}: '
            f'{response.url}. Response body: {body}'
        ) from error
    return response.json()

def iter_archive_frames(session, headers, params, page_limit=100, max_pages=None, pause_seconds=0.0):
    query = dict(params)
    query.setdefault('limit', min(int(page_limit), 100))
    query.setdefault('pagination_style', 'cursor')
    url = FRAMES_URL
    page_count = 0
    while url:
        payload = request_json(session, url, headers, params=query if url == FRAMES_URL else None)
        page_count += 1
        for frame in payload.get('results', []):
            yield frame
        if max_pages is not None and page_count >= max_pages:
            break
        url = payload.get('next')
        query = {}
        if pause_seconds:
            time.sleep(pause_seconds)

In [ ]:
def search_observation_groups(session, headers, params, observations_needed, min_frames_per_observation, page_limit=100, max_pages=5):
    query = dict(params)
    query.setdefault('limit', min(int(page_limit), 100))
    query.setdefault('pagination_style', 'cursor')
    url = FRAMES_URL
    page_count = 0
    search_frames = []
    grouped = OrderedDict()

    while url:
        print(f'  search page {page_count + 1}' + (f'/{max_pages}' if max_pages else ''), end=' ... ', flush=True)
        payload = request_json(session, url, headers, params=query if url == FRAMES_URL else None)
        page_count += 1
        page_frames = payload.get('results', [])
        search_frames.extend(page_frames)

        for frame in page_frames:
            observation_id = frame.get('observation_id')
            if observation_id is None:
                continue
            grouped.setdefault(observation_id, []).append(frame)

        qualifying = sum(1 for frames in grouped.values() if len(frames) >= min_frames_per_observation)
        print(f'{len(page_frames)} frames, {len(grouped)} observations, {qualifying} qualifying', flush=True)

        if qualifying >= observations_needed:
            break
        if max_pages is not None and page_count >= max_pages:
            break
        url = payload.get('next')
        query = {}

    return search_frames, grouped, page_count

def frame_search_params(target_name, exact_target_name=False, start=None, end=None, public=True, proposal_id=None, site_id=None, telescope_id=None, instrument_id=None, primary_optical_element=None, reduction_level=91, configuration_type='EXPOSE'):
    params = {
        'public': str(public).lower(),
        'configuration_type': configuration_type,
        'reduction_level': reduction_level,
    }
    params['target_name_exact' if exact_target_name else 'target_name'] = target_name
    optional = {
        'start': start,
        'end': end,
        'proposal_id': proposal_id,
        'site_id': site_id,
        'telescope_id': telescope_id,
        'instrument_id': instrument_id,
        'primary_optical_element': primary_optical_element,
    }
    params.update({key: value for key, value in optional.items() if value})
    return params

In [ ]:
def observation_sort_key(frames):
    dates = [frame.get('observation_date') for frame in frames if frame.get('observation_date')]
    return min(dates) if dates else ''

def select_observations(grouped_frames, count, min_frames=1, max_frames=None):
    candidates = [
        (observation_id, frames)
        for observation_id, frames in grouped_frames.items()
        if len(frames) >= min_frames and (max_frames is None or len(frames) <= max_frames)
    ]
    candidates.sort(key=lambda item: observation_sort_key(item[1]))
    return OrderedDict(candidates[:count])

def fetch_observation_frames(session, headers, observation_id, public=True, reduction_level=91, configuration_type='EXPOSE', page_limit=100, max_frames=None):
    params = {
        'observation_id': observation_id,
        'public': str(public).lower(),
        'configuration_type': configuration_type,
        'reduction_level': reduction_level,
        'limit': min(int(page_limit), 100),
        'pagination_style': 'cursor',
    }
    frames = []
    for frame in iter_archive_frames(session, headers, params=params, page_limit=page_limit):
        frames.append(frame)
        if max_frames is not None and len(frames) > max_frames:
            break
    return frames

def safe_folder_name(value):
    text = ''.join(character if character.isalnum() or character in ('-', '_', '.') else '_' for character in str(value)).strip('._-')
    return text or 'unknown'

def target_for_observation(frames):
    counts = defaultdict(int)
    for frame in frames:
        counts[str(frame.get('target_name') or 'unknown')] += 1
    return max(counts, key=counts.get) if counts else 'unknown'

def frame_id_value(frame_or_id):
    if isinstance(frame_or_id, dict):
        return frame_or_id.get('id')
    return frame_or_id

def related_frame_ids(frame):
    ids = []
    for related in frame.get('related_frames') or []:
        related_id = frame_id_value(related)
        if related_id is not None:
            ids.append(related_id)
    return ids

def fetch_frame(session, headers, frame_id):
    return request_json(session, f'{FRAMES_URL}{frame_id}/', headers)

def fetch_related_frames(session, headers, frame_id):
    payload = request_json(session, f'{FRAMES_URL}{frame_id}/related/', headers)
    if isinstance(payload, dict):
        return payload.get('results', [])
    return payload

def is_calibration_frame(frame):
    return str(frame.get('configuration_type') or '').upper() in CALIBRATION_CONFIGURATION_TYPES

def collect_related_calibration_frames(session, headers, science_frames, public=True, related_lookup_reduction_level=91, page_limit=100):
    related_ids = OrderedDict()
    related_lookup = {'related_frame_ids_from_science': {}, 'related_frame_ids_from_reduced_reference': {}, 'reduced_reference_frames': [], 'errors': []}

    for frame in science_frames:
        frame_id = frame.get('id')
        ids = related_frame_ids(frame)
        related_lookup['related_frame_ids_from_science'][str(frame_id)] = ids
        for related_id in ids:
            related_ids.setdefault(related_id, None)

    if related_lookup_reduction_level is not None:
        for observation_id in sorted({frame.get('observation_id') for frame in science_frames if frame.get('observation_id') is not None}):
            try:
                reference_frames = fetch_observation_frames(session, headers, observation_id, public, related_lookup_reduction_level, 'EXPOSE', page_limit=page_limit, max_frames=len(science_frames) + 1)
                related_lookup['reduced_reference_frames'].extend(reference_frames)
                for frame in reference_frames:
                    frame_id = frame.get('id')
                    ids = related_frame_ids(frame)
                    related_lookup['related_frame_ids_from_reduced_reference'][str(frame_id)] = ids
                    for related_id in ids:
                        related_ids.setdefault(related_id, None)
            except Exception as error:
                related_lookup['errors'].append({'stage': 'reduced_reference_lookup', 'observation_id': observation_id, 'error': str(error)})

    calibration_by_id = OrderedDict()
    for related_id in related_ids:
        try:
            related_frame = fetch_frame(session, headers, related_id)
            if is_calibration_frame(related_frame):
                calibration_by_id[related_frame['id']] = related_frame
        except Exception as error:
            related_lookup['errors'].append({'stage': 'related_frame_lookup', 'frame_id': related_id, 'error': str(error)})

    if not calibration_by_id:
        for frame in science_frames[:5]:
            frame_id = frame.get('id')
            try:
                for related_frame in fetch_related_frames(session, headers, frame_id):
                    if is_calibration_frame(related_frame):
                        calibration_by_id[related_frame['id']] = related_frame
            except Exception as error:
                related_lookup['errors'].append({'stage': 'related_endpoint_lookup', 'frame_id': frame_id, 'error': str(error)})

    calibration_frames = list(calibration_by_id.values())
    calibration_frames.sort(key=lambda frame: (str(frame.get('configuration_type') or ''), str(frame.get('observation_date') or ''), str(frame.get('id') or '')))
    return calibration_frames, related_lookup

def group_calibration_frames(calibration_frames):
    grouped = {configuration_type: [] for configuration_type in sorted(CALIBRATION_CONFIGURATION_TYPES)}
    for frame in calibration_frames:
        configuration_type = str(frame.get('configuration_type') or '').upper()
        grouped.setdefault(configuration_type, []).append(frame)
    return grouped

In [ ]:
def frame_download_url(frame):
    if frame.get('url'):
        return frame['url']
    for version in frame.get('version_set') or []:
        if version.get('url'):
            return version['url']
    return None

def frame_filename(frame):
    if frame.get('filename'):
        return frame['filename']
    basename = frame.get('basename') or frame.get('id') or 'frame'
    extension = '.fits'
    versions = frame.get('version_set') or []
    if versions and versions[0].get('extension'):
        extension = versions[0]['extension']
    return f'{basename}{extension}'

def download_frame(session, frame, destination_dir, headers, overwrite=False, timeout=120):
    download_url = frame_download_url(frame)
    filename = frame_filename(frame)
    destination = destination_dir / safe_folder_name(filename)

    if not download_url:
        return {'ok': False, 'status': 'missing_url', 'path': None, 'bytes': 0, 'error': 'Frame metadata did not include a download url.'}
    if destination.exists() and not overwrite:
        return {'ok': True, 'status': 'skipped_existing', 'path': str(destination), 'bytes': destination.stat().st_size, 'error': None}

    request_headers = headers if str(download_url).startswith(ARCHIVE_API_ROOT) else {}
    with session.get(download_url, headers=request_headers, stream=True, timeout=timeout) as response:
        response.raise_for_status()
        destination.parent.mkdir(parents=True, exist_ok=True)
        with destination.open('wb') as file:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    file.write(chunk)
    return {'ok': True, 'status': 'downloaded', 'path': str(destination), 'bytes': destination.stat().st_size, 'error': None}

def uncompressed_fits_path(path):
    lower_name = path.name.lower()
    if lower_name.endswith('.fits.fz') or lower_name.endswith('.fit.fz'):
        return path.with_name(path.name[:-3])
    return path

In [ ]:
def decompress_fits_fz(path_value, overwrite=False):
    if not path_value:
        return {'ok': False, 'status': 'missing_path', 'path': None, 'bytes': 0, 'method': None, 'error': 'No downloaded file path was provided.'}

    source = Path(path_value)
    if not source.name.lower().endswith(('.fits.fz', '.fit.fz')):
        return {'ok': True, 'status': 'not_compressed', 'path': str(source), 'bytes': source.stat().st_size if source.exists() else 0, 'method': None, 'error': None}

    destination = uncompressed_fits_path(source)
    if destination.exists() and not overwrite:
        return {'ok': True, 'status': 'skipped_existing', 'path': str(destination), 'bytes': destination.stat().st_size, 'method': None, 'error': None}

    funpack = shutil.which('funpack')
    if funpack:
        result = subprocess.run([funpack, '-O', str(destination), str(source)], capture_output=True, text=True)
        if result.returncode == 0 and destination.exists():
            return {'ok': True, 'status': 'decompressed', 'path': str(destination), 'bytes': destination.stat().st_size, 'method': 'funpack', 'error': None}

    try:
        from astropy.io import fits
        with fits.open(source) as hdul:
            hdul.writeto(destination, overwrite=True)
        return {'ok': True, 'status': 'decompressed', 'path': str(destination), 'bytes': destination.stat().st_size, 'method': 'astropy', 'error': None}
    except Exception as error:
        return {'ok': False, 'status': 'decompress_failed', 'path': None, 'bytes': 0, 'method': 'funpack_or_astropy', 'error': str(error)}

def write_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, indent=2, default=str), encoding='utf-8')

def write_frames_csv(path, frames):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=FRAME_CSV_FIELDS)
        writer.writeheader()
        for frame in frames:
            writer.writerow({field: frame.get(field) for field in FRAME_CSV_FIELDS})

def download_frame_group(session, frames, destination_dir, headers, label, decompress_fits=False, overwrite=False, keep_compressed=False):
    downloaded_files = []
    uncompressed_files = []
    download_results = []
    total_frames = len(frames)
    print(f'    downloading {total_frames} {label} FITS files into {destination_dir}')
    destination_dir.mkdir(parents=True, exist_ok=True)

    for index, frame in enumerate(frames, start=1):
        filename = frame_filename(frame)
        print(f'      [{index}/{total_frames}] {filename} ...', end=' ', flush=True)
        result = download_frame(session, frame, destination_dir, headers, overwrite)
        if result.get('path'):
            downloaded_files.append(result['path'])
        if result.get('ok'):
            size_mb = (result.get('bytes') or 0) / (1024 * 1024)
            print(f"{result['status']} ({size_mb:.1f} MB)", end='', flush=True)
            if decompress_fits:
                decompress_result = decompress_fits_fz(result.get('path'), overwrite)
                result['decompression'] = decompress_result
                if decompress_result.get('path') and decompress_result.get('path') != result.get('path'):
                    uncompressed_files.append(decompress_result['path'])
                    if not keep_compressed:
                        try:
                            Path(result['path']).unlink()
                            result['compressed_removed'] = True
                        except FileNotFoundError:
                            result['compressed_removed'] = True
                        except Exception as error:
                            result['compressed_removed'] = False
                            result['compressed_remove_error'] = str(error)
                    result['final_path'] = decompress_result['path']
                    downloaded_files[-1] = decompress_result['path']
                if decompress_result.get('ok'):
                    decompressed_mb = (decompress_result.get('bytes') or 0) / (1024 * 1024)
                    print(f"; {decompress_result['status']} -> {Path(decompress_result['path']).name} ({decompressed_mb:.1f} MB)")
                else:
                    print(f"; {decompress_result['status']}: {decompress_result.get('error')}")
            else:
                print()
        else:
            print(f"{result['status']}: {result.get('error')}")
        download_results.append({'frame_id': frame.get('id'), 'filename': filename, **result})

    return downloaded_files, uncompressed_files, download_results

In [ ]:
# Hidden e91 validation and coordinate helpers. e91 files and derived labels are written only under DEBUG_ROOT.
TARGET_COORDINATE_CACHE = {}

def normalize_target_name(value):
    text = str(value or '').strip()
    text = re.sub(r'[_]+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text

def sql_literal(value):
    return str(value).replace("'", "''")

def finite_float(value):
    try:
        result = float(value)
    except Exception:
        return None
    return result if np.isfinite(result) else None

def query_nasa_exoplanet_archive(target_name, timeout=60):
    name = normalize_target_name(target_name)
    lowered = sql_literal(name.lower())
    query = f"""
        select pl_name,hostname,ra,dec,pl_orbper,pl_tranmid
        from pscomppars
        where lower(pl_name) = '{lowered}'
           or lower(hostname) = '{lowered}'
           or lower(pl_name) like '%{lowered}%'
           or lower(hostname) like '%{lowered}%'
    """
    response = requests.get(
        NASA_EXOPLANET_TAP_SYNC_URL,
        params={'query': ' '.join(query.split()), 'format': 'json'},
        timeout=timeout,
    )
    response.raise_for_status()
    rows = response.json()
    if not rows:
        return None

    def score(row):
        pl_name = normalize_target_name(row.get('pl_name')).lower()
        host = normalize_target_name(row.get('hostname')).lower()
        value = 0
        if pl_name == name.lower():
            value += 100
        if host == name.lower():
            value += 80
        if name.lower() in pl_name:
            value += 20
        if name.lower() in host:
            value += 20
        if pl_name.endswith(' b'):
            value += 5
        return value

    rows = sorted(rows, key=score, reverse=True)
    row = rows[0]
    ra = finite_float(row.get('ra'))
    dec = finite_float(row.get('dec'))
    if ra is None or dec is None:
        return None
    return {
        'planet_name': normalize_target_name(row.get('pl_name')),
        'host_star_name': normalize_target_name(row.get('hostname')),
        'ra_deg': ra,
        'dec_deg': dec,
        'coordinate_frame': 'ICRS',
        'coordinate_source': 'NASA Exoplanet Archive',
        'ephemeris': {
            'period_days': finite_float(row.get('pl_orbper')),
            'transit_midpoint_jd': finite_float(row.get('pl_tranmid')),
        },
    }

def query_simbad_target(target_name):
    try:
        from astroquery.simbad import Simbad
        simbad = Simbad()
        table = simbad.query_object(normalize_target_name(target_name))
        if table is None or len(table) == 0:
            return None
        names = {name.lower(): name for name in table.colnames}
        ra_col = names.get('ra')
        dec_col = names.get('dec')
        if not ra_col or not dec_col:
            return None
        coord = SkyCoord(str(table[ra_col][0]), str(table[dec_col][0]), unit=(u.hourangle, u.deg), frame='icrs')
        host_name = normalize_target_name(target_name)
        return {
            'planet_name': host_name,
            'host_star_name': host_name,
            'ra_deg': float(coord.ra.deg),
            'dec_deg': float(coord.dec.deg),
            'coordinate_frame': 'ICRS',
            'coordinate_source': 'SIMBAD',
            'ephemeris': {'period_days': None, 'transit_midpoint_jd': None},
        }
    except Exception:
        return None

def resolve_target_coordinates(target_name):
    key = normalize_target_name(target_name).lower()
    if key in TARGET_COORDINATE_CACHE:
        return TARGET_COORDINATE_CACHE[key]
    resolved = query_nasa_exoplanet_archive(target_name)
    if resolved is None:
        resolved = query_simbad_target(target_name)
    TARGET_COORDINATE_CACHE[key] = resolved
    return resolved

def observation_date_key(frame):
    return str(frame.get('observation_date') or '')

def frame_jd(frame):
    value = frame.get('observation_date')
    if not value:
        return np.nan
    try:
        return float(Time(value, format='isot', scale='utc').jd)
    except Exception:
        try:
            return float(Time(value, scale='utc').jd)
        except Exception:
            return np.nan

def frame_basename_stem(frame):
    name = Path(frame_filename(frame)).name
    for suffix in ('.fits.fz', '.fit.fz', '.fits', '.fit'):
        if name.lower().endswith(suffix):
            name = name[:-len(suffix)]
            break
    return re.sub(r'-(e00|e91)$', '', name, flags=re.IGNORECASE)

def is_reduced_e91_science_frame(frame):
    return (
        str(frame.get('configuration_type') or '').upper() == 'EXPOSE'
        and int(frame.get('reduction_level') or 0) == 91
        and 'e91' in frame_filename(frame).lower()
    )

def select_science_frame_window(frames, min_frames=MIN_SCIENCE_FRAMES, max_frames=MAX_SCIENCE_FRAMES, target_identity=None):
    science_frames = [frame for frame in frames if is_reduced_e91_science_frame(frame)]
    science_frames.sort(key=observation_date_key)
    available = len(science_frames)
    if available < min_frames:
        return [], {
            'available_science_frame_count': available,
            'selected_science_frame_count': 0,
            'minimum_required_frames': min_frames,
            'maximum_allowed_frames': max_frames,
            'frame_selection_method': 'rejected_too_few_frames',
        }
    if available <= max_frames:
        return science_frames, {
            'available_science_frame_count': available,
            'selected_science_frame_count': available,
            'minimum_required_frames': min_frames,
            'maximum_allowed_frames': max_frames,
            'frame_selection_method': 'all_available_frames',
        }

    jds = np.array([frame_jd(frame) for frame in science_frames], dtype=float)
    ephemeris = (target_identity or {}).get('ephemeris') or {}
    period = finite_float(ephemeris.get('period_days'))
    t0 = finite_float(ephemeris.get('transit_midpoint_jd'))
    best_start = 0
    method = 'contiguous_window_with_fewest_temporal_gaps'

    if period and t0 and np.isfinite(jds).sum() == len(jds):
        sequence_mid = np.nanmedian(jds)
        cycle = round((sequence_mid - t0) / period)
        transit_mid = t0 + cycle * period
        centers = np.array([(jds[start] + jds[start + max_frames - 1]) / 2.0 for start in range(available - max_frames + 1)])
        best_start = int(np.nanargmin(np.abs(centers - transit_mid)))
        method = 'contiguous_window_centered_on_transit'
    elif np.isfinite(jds).sum() == len(jds):
        best = None
        for start in range(available - max_frames + 1):
            window = jds[start:start + max_frames]
            gaps = np.diff(window)
            key = (float(np.nanmax(gaps)), float(window[-1] - window[0]), start)
            if best is None or key < best:
                best = key
                best_start = start

    selected = science_frames[best_start:best_start + max_frames]
    return selected, {
        'available_science_frame_count': available,
        'selected_science_frame_count': len(selected),
        'minimum_required_frames': min_frames,
        'maximum_allowed_frames': max_frames,
        'frame_selection_method': method,
        'selected_start_observation_date': selected[0].get('observation_date') if selected else None,
        'selected_end_observation_date': selected[-1].get('observation_date') if selected else None,
    }

def select_e91_validation_subset(frames, validation_frame_count=E91_VALIDATION_FRAME_COUNT):
    frames = list(frames)
    if not frames:
        return [], {'validation_frame_count': 0, 'validation_frame_selection_method': 'no_frames'}
    requested = max(1, int(validation_frame_count or 1))
    if len(frames) <= requested:
        subset = frames
        indices = list(range(len(frames)))
        method = 'all_selected_science_frames'
    else:
        indices = sorted({int(round(value)) for value in np.linspace(0, len(frames) - 1, requested)})
        subset = [frames[index] for index in indices]
        method = 'evenly_spaced_sequence_subset'
    return subset, {
        'full_selected_science_frame_count': len(frames),
        'validation_frame_count': len(subset),
        'requested_validation_frame_count': requested,
        'validation_frame_selection_method': method,
        'validation_frame_indices': indices,
        'validation_first_observation_date': subset[0].get('observation_date') if subset else None,
        'validation_last_observation_date': subset[-1].get('observation_date') if subset else None,
    }

def build_observation_folder_name(target_identity, frames, counters):
    first = frames[0] if frames else {}
    planet = safe_folder_name(normalize_target_name((target_identity or {}).get('planet_name') or first.get('target_name') or 'unknown').replace(' ', '_'))
    date = str(first.get('observation_date') or 'unknown')[:10].replace('-', '') or 'unknown'
    site = safe_folder_name(first.get('site_id') or 'site')
    telescope = safe_folder_name(first.get('telescope_id') or 'telescope')
    base = f'{planet}_{date}_{site}_{telescope}'
    counters[base] += 1
    return f'{base}_{counters[base]:03d}'

def quiet_wcs(header):
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', message=".*'obsfix' made the change.*", category=FITSFixedWarning)
        return WCS(header)

def locate_image_hdu(hdul, require_wcs=False):
    candidates = []
    for index, hdu in enumerate(hdul):
        data = getattr(hdu, 'data', None)
        if data is None or np.ndim(data) != 2:
            continue
        header = hdu.header
        has_wcs = False
        try:
            has_wcs = bool(quiet_wcs(header).has_celestial)
        except Exception:
            has_wcs = False
        if require_wcs and not has_wcs:
            continue
        extname = str(header.get('EXTNAME') or '').upper()
        score = (100 if extname == 'SCI' else 0) + (10 if has_wcs else 0) + (1 if index == 0 else 0)
        candidates.append((score, index, hdu))
    if not candidates:
        raise RuntimeError('No two-dimensional science image HDU with required WCS was found.')
    candidates.sort(key=lambda item: item[0], reverse=True)
    _, index, hdu = candidates[0]
    return index, np.asarray(hdu.data, dtype=float), hdu.header

def validate_wcs_pixel(header, shape, target_identity):
    wcs = quiet_wcs(header).celestial
    if not wcs.has_celestial:
        raise RuntimeError('WCS(header).has_celestial is false.')
    physical_types = [str(value or '').lower() for value in getattr(wcs, 'world_axis_physical_types', [])]
    if not any('ra' in value for value in physical_types) or not any('dec' in value for value in physical_types):
        raise RuntimeError(f'Celestial WCS axes do not contain RA and Dec: {physical_types}')
    target_coord = SkyCoord(ra=target_identity['ra_deg'] * u.deg, dec=target_identity['dec_deg'] * u.deg, frame='icrs')
    x_wcs, y_wcs = wcs.world_to_pixel(target_coord)
    if not np.isfinite(x_wcs) or not np.isfinite(y_wcs):
        raise RuntimeError('WCS world_to_pixel returned non-finite coordinates.')
    height, width = shape
    if not (0 <= x_wcs < width and 0 <= y_wcs < height):
        raise RuntimeError(f'WCS coordinate is outside image bounds: x={x_wcs}, y={y_wcs}, shape={shape}.')
    roundtrip_world = wcs.pixel_to_world(x_wcs, y_wcs)
    x2, y2 = wcs.world_to_pixel(roundtrip_world)
    roundtrip_error = float(np.hypot(x2 - x_wcs, y2 - y_wcs))
    if not np.isfinite(roundtrip_error) or roundtrip_error > 1e-5:
        raise RuntimeError(f'WCS round-trip error is too large: {roundtrip_error}')
    return float(x_wcs), float(y_wcs), float(roundtrip_error), wcs

def table_column_lookup(table_hdu):
    names = list(table_hdu.columns.names or [])
    return {name.lower(): name for name in names}

def extract_sources_from_catalog(hdul):
    xy_options = [
        ('x', 'y'), ('x_image', 'y_image'), ('xwin_image', 'ywin_image'),
        ('xcentroid', 'ycentroid'), ('x_center', 'y_center'),
    ]
    candidates = []
    for index, hdu in enumerate(hdul):
        if not hasattr(hdu, 'columns') or hdu.data is None:
            continue
        lookup = table_column_lookup(hdu)
        pair = next(((lookup[x], lookup[y]) for x, y in xy_options if x in lookup and y in lookup), None)
        if pair is None:
            continue
        extname = str(hdu.header.get('EXTNAME') or '').upper()
        score = 100 if 'CAT' in extname else 0
        candidates.append((score, index, hdu, pair, lookup))
    if not candidates:
        return [], None
    candidates.sort(key=lambda item: item[0], reverse=True)
    _, index, hdu, (x_col, y_col), lookup = candidates[0]
    sources = []
    for row_index, row in enumerate(hdu.data):
        x = finite_float(row[x_col])
        y = finite_float(row[y_col])
        if x is None or y is None:
            continue
        source = {'source_id': int(row_index), 'x': x, 'y': y, 'source_method': 'e91_source_catalog'}
        for key in ['flux', 'flux_auto', 'flux_aper', 'mag', 'flags', 'flag', 'fwhm_image', 'elongation', 'ellipticity']:
            if key in lookup:
                source[key] = finite_float(row[lookup[key]])
        sources.append(source)
    return sources, {'hdu_index': index, 'x_column': x_col, 'y_column': y_col, 'source_count': len(sources)}

def detect_sources_with_daofind(data):
    clean = np.asarray(data, dtype=float)
    mean, median, std = sigma_clipped_stats(clean, sigma=3.0, maxiters=5)
    if not np.isfinite(std) or std <= 0:
        return [], {'method': 'dao_star_finder', 'source_count': 0, 'reason': 'invalid_background_std'}
    finder = DAOStarFinder(fwhm=4.0, threshold=5.0 * std)
    table = finder(clean - median)
    if table is None:
        return [], {'method': 'dao_star_finder', 'source_count': 0}
    sources = []
    for row_index, row in enumerate(table):
        source = {
            'source_id': int(row_index),
            'x': float(row['xcentroid']),
            'y': float(row['ycentroid']),
            'flux': finite_float(row['flux']),
            'sharpness': finite_float(row['sharpness']),
            'roundness1': finite_float(row['roundness1']),
            'roundness2': finite_float(row['roundness2']),
            'source_method': 'dao_star_finder',
        }
        sources.append(source)
    return sources, {'method': 'dao_star_finder', 'source_count': len(sources)}

def get_sources_for_e91(path):
    with fits.open(path, memmap=False) as hdul:
        image_index, data, header = locate_image_hdu(hdul, require_wcs=True)
        sources, catalog_info = extract_sources_from_catalog(hdul)
        method = 'e91_source_catalog'
        if not sources:
            sources, catalog_info = detect_sources_with_daofind(data)
            method = 'dao_star_finder'
        return data, header, sources, {'image_hdu_index': image_index, 'catalog': catalog_info, 'source_method': method}

def nearest_source(sources, x, y, max_distance=None):
    if not sources:
        return None, None
    distances = np.array([np.hypot(source['x'] - x, source['y'] - y) for source in sources], dtype=float)
    index = int(np.nanargmin(distances))
    distance = float(distances[index])
    if max_distance is not None and distance > max_distance:
        return None, distance
    return sources[index], distance

def parse_fits_section(value):
    if not value:
        return None
    match = re.match(r'\[\s*([+-]?\d+)\s*:\s*([+-]?\d+)\s*,\s*([+-]?\d+)\s*:\s*([+-]?\d+)\s*\]', str(value))
    if not match:
        return None
    x1, x2, y1, y2 = map(int, match.groups())
    return {'x1': x1, 'x2': x2, 'y1': y1, 'y2': y2, 'x_flip': x2 < x1, 'y_flip': y2 < y1}

def header_section_summary(header):
    keys = ['DATASEC', 'TRIMSEC', 'BIASSEC', 'CCDSEC', 'DETSEC']
    return {key: header.get(key) for key in keys if header.get(key) is not None}

def apply_pixel_transform(x, y, transform):
    transform_type = transform.get('transform_type')
    params = transform.get('transform_parameters') or {}
    if transform_type == 'identity':
        return float(x), float(y)
    if transform_type in {'constant_offset', 'crop_offset'}:
        return float(x) + float(params.get('dx', 0.0)), float(y) + float(params.get('dy', 0.0))
    if transform_type == 'axis_flip':
        x_out = float(params.get('e00_width', 0) - 1 - x) if params.get('x_flip') else float(x)
        y_out = float(params.get('e00_height', 0) - 1 - y) if params.get('y_flip') else float(y)
        return x_out, y_out
    raise RuntimeError(f'Unsupported pixel transform: {transform_type}')

def edge_distance(x, y, shape):
    height, width = shape
    return float(min(x, y, width - 1 - x, height - 1 - y))

def aperture_flux(data, x, y, radius=COMPARISON_APERTURE_RADIUS_PX):
    if not (np.isfinite(x) and np.isfinite(y)):
        return np.nan
    aperture = CircularAperture([(x, y)], r=radius)
    table = aperture_photometry(np.asarray(data, dtype=float), aperture)
    return float(table['aperture_sum'][0])

def local_peak(data, x, y, radius=3):
    y0 = max(0, int(round(y)) - radius)
    y1 = min(data.shape[0], int(round(y)) + radius + 1)
    x0 = max(0, int(round(x)) - radius)
    x1 = min(data.shape[1], int(round(x)) + radius + 1)
    if y1 <= y0 or x1 <= x0:
        return np.nan
    return float(np.nanmax(data[y0:y1, x0:x1]))

def saturation_limit(header, data):
    for key in ['SATURATE', 'SATLEVEL', 'SATURATI', 'FULLWELL']:
        value = finite_float(header.get(key))
        if value:
            return value
    return float(np.nanpercentile(data, 99.9)) / SATURATION_FRACTION

def robust_rms(values):
    array = np.asarray([value for value in values if np.isfinite(value)], dtype=float)
    if array.size < 3:
        return np.inf
    median = np.nanmedian(array)
    if not np.isfinite(median) or median == 0:
        return np.inf
    normalized = array / median
    return float(1.4826 * np.nanmedian(np.abs(normalized - np.nanmedian(normalized))))

def skycoord_for_pixel(wcs, x, y):
    try:
        return wcs.pixel_to_world(x, y)
    except Exception:
        return None

def validate_target_for_e91_frame(e91_path, target_identity):
    data, header, sources, source_info = get_sources_for_e91(e91_path)
    x_wcs, y_wcs, roundtrip_error, wcs = validate_wcs_pixel(header, data.shape, target_identity)
    source, distance = nearest_source(sources, x_wcs, y_wcs, TARGET_MATCH_MAX_DISTANCE_PX)
    if source is None:
        raise RuntimeError(f'No detected source within {TARGET_MATCH_MAX_DISTANCE_PX}px of target WCS prediction; nearest={distance}.')
    transform = {'transform_type': 'identity', 'transform_parameters': {}, 'debug': {'science_frame_type': 'e91'}}
    x_science, y_science = apply_pixel_transform(source['x'], source['y'], transform)
    return {
        'x_wcs': x_wcs,
        'y_wcs': y_wcs,
        'x_target_e91': float(source['x']),
        'y_target_e91': float(source['y']),
        'x_target_science': x_science,
        'y_target_science': y_science,
        'wcs_match_distance_px': float(distance),
        'matching_method': source.get('source_method') or source_info.get('source_method'),
        'wcs_roundtrip_error_px': roundtrip_error,
        'transform': transform,
        'source_info': source_info,
        'source_count': len(sources),
        'validation_status': 'valid',
    }, {'data': data, 'header': header, 'sources': sources, 'wcs': wcs}

def transform_signature(transform):
    params = transform.get('transform_parameters') or {}
    rounded = {key: (round(value, 6) if isinstance(value, float) else value) for key, value in params.items()}
    return json.dumps({'transform_type': transform.get('transform_type'), 'transform_parameters': rounded}, sort_keys=True)

def evaluate_comparison_candidates(valid_frames, common_transform):
    if not valid_frames:
        return [], []
    base = valid_frames[0]
    base_data = base['analysis']['data']
    base_header = base['analysis']['header']
    base_sources = base['analysis']['sources']
    base_wcs = base['analysis']['wcs']
    base_target = base['target']
    shape = base_data.shape
    sat_limit = saturation_limit(base_header, base_data)
    single_frame_mode = len(valid_frames) == 1
    target_fluxes = [aperture_flux(frame['analysis']['data'], frame['target']['x_target_e91'], frame['target']['y_target_e91']) for frame in valid_frames]
    target_median_flux = float(np.nanmedian(target_fluxes)) if target_fluxes else np.nan
    all_candidates = []

    for source in base_sources:
        x = float(source['x'])
        y = float(source['y'])
        distance_to_target = float(np.hypot(x - base_target['x_target_e91'], y - base_target['y_target_e91']))
        edge_px = edge_distance(x, y, shape)
        if distance_to_target < MIN_TARGET_COMPARISON_DISTANCE_PX:
            continue
        if edge_px < MIN_EDGE_DISTANCE_PX:
            continue
        neighbor_distances = [np.hypot(x - other['x'], y - other['y']) for other in base_sources if other is not source]
        nearest_neighbor = float(np.nanmin(neighbor_distances)) if neighbor_distances else np.inf
        if nearest_neighbor < MIN_COMPARISON_ISOLATION_PX:
            continue
        source_sky = skycoord_for_pixel(base_wcs, x, y)
        if source_sky is None:
            continue
        per_frame = []
        fluxes = []
        peaks = []
        backgrounds = []
        airmasses = []
        for frame in valid_frames:
            data = frame['analysis']['data']
            header = frame['analysis']['header']
            wcs = frame['analysis']['wcs']
            try:
                x_pred, y_pred = wcs.world_to_pixel(source_sky)
            except Exception:
                continue
            matched, match_distance = nearest_source(frame['analysis']['sources'], x_pred, y_pred, TARGET_MATCH_MAX_DISTANCE_PX)
            if matched is None:
                continue
            flux = aperture_flux(data, matched['x'], matched['y'])
            peak = local_peak(data, matched['x'], matched['y'])
            x_science, y_science = apply_pixel_transform(matched['x'], matched['y'], common_transform)
            science_filename = Path(frame.get('science_path') or frame['e91_path']).name
            per_frame.append({
                'science_filename': science_filename,
                'e91_filename': Path(frame['e91_path']).name,
                'x_e91': float(matched['x']),
                'y_e91': float(matched['y']),
                'x_science': x_science,
                'y_science': y_science,
                'match_distance_px': float(match_distance),
                'flux': flux,
            })
            fluxes.append(flux)
            peaks.append(peak)
            backgrounds.append(float(sigma_clipped_stats(data, sigma=3.0, maxiters=3)[1]))
            airmasses.append(finite_float(header.get('AIRMASS')) or np.nan)
        detection_fraction = len(per_frame) / len(valid_frames)
        median_flux = float(np.nanmedian(fluxes)) if fluxes else np.nan
        target_flux_ratio = float(median_flux / target_median_flux) if np.isfinite(median_flux) and np.isfinite(target_median_flux) and target_median_flux > 0 else np.nan
        brightness_similarity_score = float(abs(np.log10(target_flux_ratio))) if np.isfinite(target_flux_ratio) and target_flux_ratio > 0 else np.inf
        rms = 0.0 if single_frame_mode and len(fluxes) == 1 else robust_rms(fluxes)
        saturated = bool(np.nanmax(peaks) >= SATURATION_FRACTION * sat_limit) if peaks else False
        background_corr = None
        if len(fluxes) >= 5 and np.nanstd(backgrounds) > 0:
            background_corr = float(np.corrcoef(np.asarray(fluxes, dtype=float), np.asarray(backgrounds, dtype=float))[0, 1])
        airmass_corr = None
        if len(fluxes) >= 5 and np.isfinite(airmasses).sum() >= 5 and np.nanstd(airmasses) > 0:
            airmass_corr = float(np.corrcoef(np.asarray(fluxes, dtype=float), np.asarray(airmasses, dtype=float))[0, 1])
        x_science, y_science = apply_pixel_transform(x, y, common_transform)
        candidate = {
            'comparison_star_id': int(source.get('source_id', len(all_candidates) + 1)),
            'x_e91': x,
            'y_e91': y,
            'x_science': x_science,
            'y_science': y_science,
            'median_flux': median_flux,
            'target_median_flux': target_median_flux,
            'target_flux_ratio': target_flux_ratio,
            'brightness_similarity_score': brightness_similarity_score,
            'robust_rms': rms,
            'detection_fraction': float(detection_fraction),
            'saturated': saturated,
            'edge_distance_px': edge_px,
            'nearest_neighbor_distance_px': nearest_neighbor,
            'airmass_correlation': airmass_corr,
            'background_correlation': background_corr,
            'valid_measurement_count': len(per_frame),
            'validation_mode': 'single_frame_static_reference' if single_frame_mode else 'multi_frame_stability_reference',
            'per_frame': per_frame,
        }
        candidate['accepted'] = (
            detection_fraction >= MIN_FRAME_DETECTION_FRACTION
            and np.isfinite(median_flux)
            and median_flux > 0
            and np.isfinite(target_flux_ratio)
            and MIN_COMPARISON_TARGET_FLUX_RATIO <= target_flux_ratio <= MAX_COMPARISON_TARGET_FLUX_RATIO
            and np.isfinite(rms)
            and rms <= MAX_COMPARISON_ROBUST_RMS
            and not saturated
        )
        all_candidates.append(candidate)

    accepted = [candidate for candidate in all_candidates if candidate['accepted']]
    accepted.sort(key=lambda item: (item['robust_rms'], item['brightness_similarity_score'], -item['detection_fraction'], -item['median_flux']))
    accepted = accepted[:MAX_COMPARISON_STARS]
    for rank, candidate in enumerate(accepted, start=1):
        candidate['rank'] = rank
        candidate['comparison_star_id'] = rank
    for candidate in all_candidates:
        candidate.setdefault('rank', None)
    return accepted, all_candidates

def write_frame_coordinate_table(path, valid_frames, comparison_stars):
    fieldnames = ['science_filename', 'e91_filename', 'target_x_e91', 'target_y_e91']
    for index in range(1, min(len(comparison_stars), MAX_COMPARISON_STARS) + 1):
        fieldnames.extend([f'comparison_{index}_x_e91', f'comparison_{index}_y_e91'])
    fieldnames.append('validation_status')
    rows = []
    for frame in valid_frames:
        row = {
            'science_filename': Path(frame.get('science_path') or frame['e91_path']).name,
            'e91_filename': Path(frame['e91_path']).name,
            'target_x_e91': frame['target']['x_target_e91'],
            'target_y_e91': frame['target']['y_target_e91'],
            'validation_status': frame['target']['validation_status'],
        }
        for index, star in enumerate(comparison_stars, start=1):
            match = next((item for item in star.get('per_frame', []) if item['science_filename'] == row['science_filename']), None)
            row[f'comparison_{index}_x_e91'] = match.get('x_e91') if match else None
            row[f'comparison_{index}_y_e91'] = match.get('y_e91') if match else None
        rows.append(row)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

def generate_visual_audits(debug_dir, valid_frames, comparison_stars, all_candidates):
    if not valid_frames:
        return
    import matplotlib.pyplot as plt
    frame = valid_frames[0]
    data = frame['analysis']['data']
    target = frame['target']
    verification_dir = debug_dir / 'verification'
    verification_dir.mkdir(parents=True, exist_ok=True)
    vmin, vmax = np.nanpercentile(data, [5, 99.5])

    plt.figure(figsize=(10, 10))
    plt.imshow(data, origin='lower', cmap='gray', vmin=vmin, vmax=vmax)
    plt.scatter([target['x_target_e91']], [target['y_target_e91']], marker='+', s=180, c='red', label='target')
    if comparison_stars:
        plt.scatter([star['x_e91'] for star in comparison_stars], [star['y_e91'] for star in comparison_stars], marker='o', s=80, facecolors='none', edgecolors='lime', label='accepted comparison')
        for star in comparison_stars:
            plt.text(star['x_e91'] + 8, star['y_e91'] + 8, f"{star['comparison_star_id']}: ({star['x_e91']:.1f}, {star['y_e91']:.1f})", color='lime', fontsize=8)
    rejected = [candidate for candidate in all_candidates if not candidate.get('accepted')]
    if rejected:
        plt.scatter([candidate['x_e91'] for candidate in rejected[:200]], [candidate['y_e91'] for candidate in rejected[:200]], marker='x', s=20, c='orange', alpha=0.5, label='rejected candidate')
    plt.text(target['x_target_e91'] + 8, target['y_target_e91'] + 8, f"target ({target['x_target_e91']:.1f}, {target['y_target_e91']:.1f})", color='red', fontsize=9)
    plt.legend(loc='best')
    plt.tight_layout()
    plt.savefig(verification_dir / 'target_and_comparison_overlay.png', dpi=150)
    plt.close()

    x = int(round(target['x_target_e91']))
    y = int(round(target['y_target_e91']))
    radius = 50
    y0, y1 = max(0, y - radius), min(data.shape[0], y + radius + 1)
    x0, x1 = max(0, x - radius), min(data.shape[1], x + radius + 1)
    crop = data[y0:y1, x0:x1]
    plt.figure(figsize=(5, 5))
    plt.imshow(crop, origin='lower', cmap='gray', vmin=np.nanpercentile(crop, 5), vmax=np.nanpercentile(crop, 99.5))
    plt.scatter([target['x_target_e91'] - x0], [target['y_target_e91'] - y0], marker='+', s=180, c='red')
    plt.title(f"target ({target['x_target_e91']:.2f}, {target['y_target_e91']:.2f})")
    plt.tight_layout()
    plt.savefig(verification_dir / 'target_crop.png', dpi=150)
    plt.close()

def validate_observation_ground_truth(science_frames, science_download_results, target_identity, observation_dir, debug_dir):
    debug_dir.mkdir(parents=True, exist_ok=True)
    science_paths = {result.get('frame_id'): result.get('final_path') or result.get('path') for result in science_download_results if result.get('ok')}
    valid_frames = []
    rejected_frames = []

    for frame in science_frames:
        science_path = science_paths.get(frame.get('id'))
        if not science_path:
            rejected_frames.append({'science_filename': frame_filename(frame), 'reason': 'missing_downloaded_e91_science_file'})
            continue
        try:
            target, analysis = validate_target_for_e91_frame(science_path, target_identity)
            valid_frames.append({'science_frame': frame, 'e91_frame': frame, 'science_path': science_path, 'e91_path': science_path, 'target': target, 'analysis': analysis})
        except Exception as error:
            rejected_frames.append({'science_filename': frame_filename(frame), 'reason': str(error)})

    target_validation_fraction = len(valid_frames) / max(len(science_frames), 1)
    transform_counts = defaultdict(int)
    for frame in valid_frames:
        transform_counts[transform_signature(frame['target']['transform'])] += 1
    common_transform = None
    if transform_counts:
        common_signature = max(transform_counts, key=transform_counts.get)
        common_transform = json.loads(common_signature)
    transform_consistent = bool(common_transform) and transform_counts[json.dumps(common_transform, sort_keys=True)] == len(valid_frames)

    accepted_comparisons = []
    all_candidates = []
    if target_validation_fraction >= MIN_TARGET_VALIDATION_FRACTION and transform_consistent:
        accepted_comparisons, all_candidates = evaluate_comparison_candidates(valid_frames, common_transform)

    accepted = (
        target_validation_fraction >= MIN_TARGET_VALIDATION_FRACTION
        and bool(valid_frames)
        and transform_consistent
        and len(accepted_comparisons) >= MIN_COMPARISON_STARS
    )
    validation_mode = 'single_e91_science_frame_reference' if len(science_frames) == 1 else 'multi_e91_science_frame_sequence_reference'

    target_truth = {
        'target_catalog_coordinate': {key: target_identity.get(key) for key in ['planet_name', 'host_star_name', 'ra_deg', 'dec_deg', 'coordinate_frame', 'coordinate_source']},
        'target_validation_fraction': target_validation_fraction,
        'minimum_target_validation_fraction': MIN_TARGET_VALIDATION_FRACTION,
        'transform_consistent': transform_consistent,
        'transform': common_transform,
        'validation_mode': validation_mode,
        'accepted': accepted,
        'frame_target_coordinates': [
            {
                'science_filename': Path(frame['science_path']).name,
                'e91_filename': Path(frame['e91_path']).name,
                **{key: frame['target'][key] for key in ['x_wcs', 'y_wcs', 'x_target_e91', 'y_target_e91', 'x_target_science', 'y_target_science', 'wcs_match_distance_px', 'matching_method', 'wcs_roundtrip_error_px', 'validation_status']},
            }
            for frame in valid_frames
        ],
        'rejected_frames': rejected_frames,
    }
    comparison_truth = {
        'accepted': accepted,
        'minimum_comparison_stars': MIN_COMPARISON_STARS,
        'maximum_comparison_stars': MAX_COMPARISON_STARS,
        'validation_mode': validation_mode,
        'accepted_comparison_stars': accepted_comparisons,
        'candidate_quality_metrics': all_candidates,
    }
    debug_log = {
        'accepted': accepted,
        'checked_e91_science_frame_count': len(science_frames),
        'validation_mode': validation_mode,
        'valid_target_frame_count': len(valid_frames),
        'target_validation_fraction': target_validation_fraction,
        'comparison_star_count': len(accepted_comparisons),
        'rejected_frames': rejected_frames,
        'acceptance_criteria': {
            'min_science_frames': MIN_SCIENCE_FRAMES,
            'max_science_frames': MAX_SCIENCE_FRAMES,
            'target_match_max_distance_px': TARGET_MATCH_MAX_DISTANCE_PX,
            'min_target_validation_fraction': MIN_TARGET_VALIDATION_FRACTION,
            'min_comparison_stars': MIN_COMPARISON_STARS,
        },
    }

    write_json(debug_dir / 'target_ground_truth.json', target_truth)
    write_json(debug_dir / 'comparison_star_ground_truth.json', comparison_truth)
    write_frame_coordinate_table(debug_dir / 'frame_coordinates.csv', valid_frames, accepted_comparisons)
    write_json(debug_dir / 'debug_log.json', debug_log)
    if valid_frames:
        generate_visual_audits(debug_dir, valid_frames, accepted_comparisons, all_candidates)
    return accepted, debug_log


In [ ]:
def scrape_lco_exoplanet_observations(targets, output_root, debug_root, observations_per_target=1, min_frames_per_observation=MIN_SCIENCE_FRAMES, max_frames_per_observation=MAX_SCIENCE_FRAMES, max_total_observations=20, validation_e91_frame_count=E91_VALIDATION_FRAME_COUNT, exact_target_name=False, start=None, end=None, public=True, proposal_id=None, site_id=None, telescope_id=None, instrument_id=None, primary_optical_element=None, reduction_level=91, configuration_type='EXPOSE', search_page_limit=25, max_search_pages=5, api_token=None, download_fits=True, decompress_fits=False, overwrite=False, related_lookup_reduction_level=91, require_calibration_frames=False, keep_compressed=False):
    output_root = Path(output_root).expanduser().resolve()
    debug_root = Path(debug_root).expanduser().resolve()
    session = requests.Session()
    headers = archive_headers(api_token)
    observation_name_counters = defaultdict(int)
    summary = {
        'archive_api_root': ARCHIVE_API_ROOT,
        'output_root': str(output_root),
        'debug_root': str(debug_root),
        'science_reduction_level': reduction_level,
        'related_lookup_reduction_level': related_lookup_reduction_level,
        'require_calibration_frames': require_calibration_frames,
        'keep_compressed': keep_compressed,
        'min_science_frames': min_frames_per_observation,
        'max_science_frames': max_frames_per_observation,
        'validation_e91_frame_count': validation_e91_frame_count,
        'max_total_observations': max_total_observations,
        'complete_observation_count': 0,
        'targets': {},
    }

    for target in targets:
        if max_total_observations is not None and summary['complete_observation_count'] >= max_total_observations:
            print(f"Reached max_total_observations={max_total_observations}; stopping target search.")
            break
        print(f"Searching LCO archive for {target!r}...", flush=True)
        params = frame_search_params(target, exact_target_name, start, end, public, proposal_id, site_id, telescope_id, instrument_id, primary_optical_element, reduction_level, configuration_type)
        remaining_total = None if max_total_observations is None else max_total_observations - summary['complete_observation_count']
        target_needed = observations_per_target if remaining_total is None else min(observations_per_target, remaining_total)
        candidate_count = max(target_needed * 5, target_needed)
        target_complete_count = 0
        target_summary = {
            'query': params,
            'search_page_limit': search_page_limit,
            'max_search_pages': max_search_pages,
            'min_science_frames': min_frames_per_observation,
            'max_science_frames': max_frames_per_observation,
            'validation_e91_frame_count': validation_e91_frame_count,
            'max_total_observations': max_total_observations,
            'target_needed': target_needed,
            'search_pages_read': 0,
            'frames_found': 0,
            'observations_found': 0,
            'observations_selected': [],
            'observations_skipped': [],
            'error': None,
        }

        try:
            search_frames, grouped, search_pages_read = search_observation_groups(session, headers, params, candidate_count, min_frames_per_observation, search_page_limit, max_search_pages)
        except Exception as error:
            target_summary['error'] = str(error)
            summary['targets'][target] = target_summary
            print(f'  skipped target after archive search error: {error}', flush=True)
            continue

        selected = select_observations(grouped, candidate_count, min_frames_per_observation, None)
        target_summary.update({'search_pages_read': search_pages_read, 'frames_found': len(search_frames), 'observations_found': len(grouped)})

        for observation_id, initial_frames in selected.items():
            if target_complete_count >= target_needed:
                break
            if max_total_observations is not None and summary['complete_observation_count'] >= max_total_observations:
                break

            all_frames = fetch_observation_frames(session, headers, observation_id, public, reduction_level, configuration_type, max_frames=None)
            all_frames = [frame for frame in all_frames if is_reduced_e91_science_frame(frame)]
            all_frames.sort(key=observation_date_key)
            observation_target = target_for_observation(all_frames or initial_frames)
            target_identity = resolve_target_coordinates(observation_target)
            if not target_identity:
                target_summary['observations_skipped'].append({'observation_id': observation_id, 'reason': 'unresolved_target_coordinates', 'target': observation_target})
                print(f'  skipped observation_id={observation_id}: could not resolve target coordinates for {observation_target!r}')
                continue

            frames, frame_selection = select_science_frame_window(all_frames, min_frames_per_observation, max_frames_per_observation, target_identity)
            if not frames:
                target_summary['observations_skipped'].append({'observation_id': observation_id, 'reason': frame_selection['frame_selection_method'], **frame_selection})
                print(f"  skipped observation_id={observation_id}: {frame_selection['available_science_frame_count']} reduced e91 science frames")
                continue
            validation_frames, validation_selection = select_e91_validation_subset(frames, validation_e91_frame_count)
            frame_selection['e91_science_validation'] = validation_selection

            observation_name = build_observation_folder_name(target_identity, frames, observation_name_counters)
            observation_dir = output_root / observation_name
            debug_dir = debug_root / observation_name
            metadata_dir = debug_dir / METADATA_SUBDIR
            debug_dir.mkdir(parents=True, exist_ok=True)

            calibration_frames = []
            calibration_groups = {}
            calibration_counts = {}
            related_lookup = {'skipped': True, 'reason': 'calibration downloads disabled; e91 science frames are already reduced'}
            print('    calibration lookup/download disabled; using reduced e91 science frames only')

            observation_dir.mkdir(parents=True, exist_ok=True)
            metadata_dir.mkdir(parents=True, exist_ok=True)

            science_downloaded_files = []
            science_uncompressed_files = []
            science_download_results = []
            calibration_downloaded_files = []
            calibration_uncompressed_files = []
            calibration_download_results = []
            if download_fits:
                science_downloaded_files, science_uncompressed_files, science_download_results = download_frame_group(session, frames, observation_dir, headers, 'reduced e91 science', decompress_fits, overwrite, keep_compressed)
            else:
                print(f'    download_fits=False; wrote metadata only for {len(frames)} reduced e91 science frames')

            successful_science_count = sum(1 for result in science_download_results if result.get('ok')) if download_fits else len(frames)
            if successful_science_count < min_frames_per_observation or successful_science_count > max_frames_per_observation:
                write_json(debug_dir / 'debug_log.json', {'accepted': False, 'reason': 'science_download_count_out_of_range', 'successful_science_count': successful_science_count, 'frame_selection': frame_selection})
                if observation_dir.exists():
                    shutil.rmtree(observation_dir)
                target_summary['observations_skipped'].append({'observation_id': observation_id, 'observation_name': observation_name, 'reason': 'science_download_count_out_of_range', 'science_downloaded_file_count': successful_science_count, **frame_selection})
                print(f'  skipped observation_id={observation_id}: only {successful_science_count} reduced e91 science files saved')
                continue

            accepted, validation_log = validate_observation_ground_truth(
                science_frames=validation_frames,
                science_download_results=science_download_results,
                target_identity=target_identity,
                observation_dir=observation_dir,
                debug_dir=debug_dir,
            )

            science_complete = (not download_fits) or (successful_science_count == len(frames) and all(result.get('ok') for result in science_download_results))
            calibration_complete = True
            observation_complete = science_complete and calibration_complete and accepted

            write_json(metadata_dir / 'frames.json', frames)
            write_frames_csv(metadata_dir / 'frames.csv', frames)
            write_json(metadata_dir / 'science_frames.json', frames)
            write_frames_csv(metadata_dir / 'science_frames.csv', frames)
            write_json(metadata_dir / 'calibration_frames.json', calibration_frames)
            write_frames_csv(metadata_dir / 'calibration_frames.csv', calibration_frames)
            write_json(metadata_dir / 'related_lookup.json', related_lookup)
            write_json(metadata_dir / 'download_results.json', {'science': science_download_results, 'calibration': calibration_download_results})
            write_json(metadata_dir / 'observation.json', {
                'target': observation_target,
                'target_identity': {key: target_identity.get(key) for key in ['planet_name', 'host_star_name']},
                'observation_id': observation_id,
                'observation_name': observation_name,
                'science_reduction_level': reduction_level,
                'science_frame_count': len(frames),
                'science_downloaded_file_count': successful_science_count,
                'science_downloaded_files': science_downloaded_files,
                'science_uncompressed_file_count': len(science_uncompressed_files),
                'science_uncompressed_files': science_uncompressed_files,
                'calibration_frame_count': len(calibration_frames),
                'calibration_counts': calibration_counts,
                'calibration_downloaded_file_count': len(calibration_downloaded_files),
                'calibration_downloaded_files': calibration_downloaded_files,
                'calibration_uncompressed_file_count': len(calibration_uncompressed_files),
                'calibration_uncompressed_files': calibration_uncompressed_files,
                'frame_selection': frame_selection,
                'science_complete': science_complete,
                'calibration_complete': calibration_complete,
                'ground_truth_validation_complete': accepted,
                'complete': observation_complete,
                'first_observation_date': frames[0].get('observation_date') if frames else None,
                'last_observation_date': frames[-1].get('observation_date') if frames else None,
                'output_dir': str(observation_dir),
                'metadata_dir': str(metadata_dir),
                'science_dir': str(observation_dir),
                'calibration_dirs': {},
            })

            selected_summary = {
                'observation_id': observation_id,
                'observation_name': observation_name,
                'target': observation_target,
                'science_frame_count': len(frames),
                'science_downloaded_file_count': successful_science_count,
                'calibration_frame_count': len(calibration_frames),
                'calibration_counts': calibration_counts,
                'calibration_downloaded_file_count': len(calibration_downloaded_files),
                'complete': observation_complete,
                'output_dir': str(observation_dir),
                'debug_dir': str(debug_dir),
                'metadata_dir': str(metadata_dir),
                'frame_selection': frame_selection,
            }
            target_summary['observations_selected'].append(selected_summary)
            if observation_complete:
                target_complete_count += 1
                summary['complete_observation_count'] += 1
            else:
                if observation_dir.exists():
                    shutil.rmtree(observation_dir)
                reason = 'ground_truth_validation_failed' if not accepted else 'download_incomplete'
                target_summary['observations_skipped'].append({'observation_id': observation_id, 'observation_name': observation_name, 'reason': reason, 'validation_log': validation_log, **frame_selection})
            print(f"  observation_id={observation_id}: {len(frames)} reduced e91 science frames, calibration downloads disabled, accepted={observation_complete}, total_complete={summary['complete_observation_count']}")

        summary['targets'][target] = target_summary

    write_json(debug_root / METADATA_SUBDIR / 'scrape_summary.json', summary)
    return summary


In [ ]:
# Configuration. This writes directly to Google Drive.
OBSERVATIONS_ROOT = Path('/content/drive/MyDrive/ExoAgent/exoagent_lco_observations')
DEBUG_ROOT = Path('/content/drive/MyDrive/ExoAgent/exoagent_lco_debug')

# Lightweight test mode processes one accepted observation first.
LIGHTWEIGHT_TEST_MODE = False
RUN_SCRAPER = True

TARGETS = DEFAULT_TARGETS[:1] if LIGHTWEIGHT_TEST_MODE else DEFAULT_TARGETS
OBSERVATIONS_PER_TARGET = 1
MAX_TOTAL_OBSERVATIONS = 1 if LIGHTWEIGHT_TEST_MODE else 30
MIN_FRAMES_PER_OBSERVATION = MIN_SCIENCE_FRAMES
MAX_FRAMES_PER_OBSERVATION = MAX_SCIENCE_FRAMES
VALIDATION_E91_FRAME_COUNT = E91_VALIDATION_FRAME_COUNT
SCIENCE_REDUCTION_LEVEL = 91
RELATED_LOOKUP_REDUCTION_LEVEL = 91
REQUIRE_CALIBRATION_FRAMES = False
SEARCH_PAGE_LIMIT = 25
MAX_SEARCH_PAGES = 5

# Produces normal FITS files from .fits.fz downloads for EXOTIC.
# Set this to False if Drive storage becomes the bottleneck.
DECOMPRESS_FITS_FZ = True
KEEP_COMPRESSED_DOWNLOADS = False

# Optional filters. ISO datetime strings can reduce downloads substantially.
START = None
END = None
FILTER = None


In [ ]:
if RUN_SCRAPER:
    summary = scrape_lco_exoplanet_observations(
        targets=TARGETS,
        output_root=OBSERVATIONS_ROOT,
        debug_root=DEBUG_ROOT,
        observations_per_target=OBSERVATIONS_PER_TARGET,
        min_frames_per_observation=MIN_FRAMES_PER_OBSERVATION,
        max_frames_per_observation=MAX_FRAMES_PER_OBSERVATION,
        validation_e91_frame_count=VALIDATION_E91_FRAME_COUNT,
        max_total_observations=MAX_TOTAL_OBSERVATIONS,
        exact_target_name=False,
        start=START,
        end=END,
        public=True,
        primary_optical_element=FILTER,
        reduction_level=SCIENCE_REDUCTION_LEVEL,
        configuration_type='EXPOSE',
        search_page_limit=SEARCH_PAGE_LIMIT,
        max_search_pages=MAX_SEARCH_PAGES,
        api_token=os.getenv('LCO_ARCHIVE_TOKEN'),
        download_fits=True,
        decompress_fits=DECOMPRESS_FITS_FZ,
        overwrite=False,
        related_lookup_reduction_level=RELATED_LOOKUP_REDUCTION_LEVEL,
        require_calibration_frames=REQUIRE_CALIBRATION_FRAMES,
        keep_compressed=KEEP_COMPRESSED_DOWNLOADS,
    )
    print('Done. Summary written to:', DEBUG_ROOT / METADATA_SUBDIR / 'scrape_summary.json')
else:
    print('Scraper is configured but not running. Set RUN_SCRAPER = True to run lightweight test mode first.')

Streaming output truncated to the last 5000 lines.
      [258/300] ogg2m001-ep03-20250121-0898-e91.fits.fz ... downloaded (9.0 MB); decompressed -> ogg2m001-ep03-20250121-0898-e91.fits (9.0 MB)
      [259/300] ogg2m001-ep02-20250121-0624-e91.fits.fz ... downloaded (9.0 MB); decompressed -> ogg2m001-ep02-20250121-0624-e91.fits (9.0 MB)
      [260/300] ogg2m001-ep04-20250121-1174-e91.fits.fz ... downloaded (9.0 MB); decompressed -> ogg2m001-ep04-20250121-1174-e91.fits (9.0 MB)
      [261/300] ogg2m001-ep05-20250121-1355-e91.fits.fz ... downloaded (9.2 MB); decompressed -> ogg2m001-ep05-20250121-1355-e91.fits (9.2 MB)
      [262/300] ogg2m001-ep03-20250121-0899-e91.fits.fz ... downloaded (9.0 MB); decompressed -> ogg2m001-ep03-20250121-0899-e91.fits (9.0 MB)
      [263/300] ogg2m001-ep04-20250121-1175-e91.fits.fz ... downloaded (9.0 MB); decompressed -> ogg2m001-ep04-20250121-1175-e91.fits (9.0 MB)
      [264/300] ogg2m001-ep05-20250121-1356-e91.fits.fz ... downloaded (9.1 MB); decompress